# Heart Disease Prediction

Predict the presence of heart disease from 13 clinical features (UCI Cleveland dataset).
Educational project, not a diagnostic tool.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay,
                             recall_score)

RANDOM_STATE = 42

## 2. Data

In [ ]:
df = pd.read_csv("data/heart.csv")   # relative path, works outside Colab
print(df.shape)
df.head()

In [ ]:
print(df.isna().sum().sum(), "missing values")
print(df["target"].value_counts(normalize=True).round(3))

## 3. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(x="target", data=df, ax=axes[0])
axes[0].set_title("Class balance")
sns.heatmap(df.corr(), cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Feature correlations")
plt.tight_layout(); plt.show()

## 4. Train/test split (stratified)

In [ ]:
X = df.drop(columns="target")
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

## 5. Model

Scaling lives inside a `Pipeline`. This fixes the earlier `ConvergenceWarning`
and guarantees the app applies the same preprocessing as training.

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_proba), 3))
print()
print(classification_report(y_test, y_pred))

## 6. Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=["Healthy", "Disease"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix")
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title("ROC curve")
plt.tight_layout(); plt.show()

### Cross-validation

With only 61 test rows, a single split is noisy. 5-fold stratified CV gives a steadier estimate.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
print(f"CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

## 7. Model comparison

In [ ]:
candidates = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC(probability=True, random_state=RANDOM_STATE)),
}

rows = []
for name, est in candidates.items():
    acc = cross_val_score(est, X, y, cv=cv, scoring="accuracy")
    rec = cross_val_score(est, X, y, cv=cv, scoring="recall")
    auc = cross_val_score(est, X, y, cv=cv, scoring="roc_auc")
    rows.append({"Model": name,
                 "Accuracy": f"{acc.mean():.3f} ± {acc.std():.3f}",
                 "Recall (disease)": f"{rec.mean():.3f} ± {rec.std():.3f}",
                 "ROC-AUC": f"{auc.mean():.3f} ± {auc.std():.3f}"})
pd.DataFrame(rows)

## 8. Interpretation

Coefficients are on standardized features, so their magnitudes are directly comparable.
Positive values push the prediction toward heart disease.

In [ ]:
coefs = pd.Series(model.named_steps["logisticregression"].coef_[0], index=X.columns).sort_values()
coefs.plot(kind="barh", figsize=(7, 5), title="Logistic regression coefficients")
plt.axvline(0, color="k", lw=0.8); plt.tight_layout(); plt.show()

## 9. Threshold analysis

For screening, missing a sick patient is worse than a false alarm. Lowering the
threshold trades precision for recall.

In [ ]:
for t in [0.3, 0.4, 0.5, 0.6]:
    pred_t = (y_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_t).ravel()
    print(f"threshold={t:.1f}  recall={recall_score(y_test, pred_t):.2f}  missed cases (FN)={fn}  false alarms (FP)={fp}")

## 10. Save the full pipeline

In [ ]:
joblib.dump(model, "models/logistic_model.pkl")
print("Saved pipeline (scaler + model) to models/logistic_model.pkl")